In [1]:
import torch
from datasets import Dataset

In [2]:
from translation.translation_dataloader import TranslationDataset

train_data = []
val_data = []
test_data = []

data = TranslationDataset(files=("../../../data/translation/dataset_part1.txt", "../../../data/translation/dataset_part2.txt"))
train_iter, val_iter, test_iter = torch.utils.data.random_split(data, [0.9, 0.05, 0.05])

for sample, target in train_iter:
    train_data.append({"translation": {"jpm": sample, "pl": target}})
    
for sample, target in val_iter:
    val_data.append({"translation": {"jpm": sample, "pl": target}})


In [3]:
import pandas as pd

df = pd.DataFrame(train_data)
train_dataset = Dataset.from_pandas(df)
df = pd.DataFrame(val_data)
val_dataset = Dataset.from_pandas(df)


In [4]:
from transformers import AutoTokenizer

checkpoint = "sdadas/polish-bart-base"
tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large", return_tensors="pt")

E:\Python\Projects\PSL-Translator\.venv\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [5]:
source_lang = "jpm"
target_lang = "pl"


def preprocess_function(example):
    inputs = example["translation"][source_lang]
    targets = example["translation"][target_lang]
    model_inputs = tokenizer(inputs, max_length=128, truncation=True)
    labels = tokenizer(targets, max_length=128, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    model_inputs["translation"] = example["translation"]
    return model_inputs


train_data = train_dataset.map(preprocess_function)
val_data = val_dataset.map(preprocess_function)

Map:   0%|          | 0/29955 [00:00<?, ? examples/s]

Map:   0%|          | 0/1664 [00:00<?, ? examples/s]

In [6]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, \
    AutoModelForSeq2SeqLM
from transformers import DataCollatorForSeq2Seq


model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint, device_map="auto")

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True)

E:\Python\Projects\PSL-Translator\.venv\lib\site-packages\accelerate\utils\modeling.py:856: UserWarning: expandable_segments not supported on this platform (Triggered internally at ..\c10/cuda/CUDAAllocatorConfig.h:28.)
  _ = torch.tensor([0], device=i)


In [7]:
training_args = Seq2SeqTrainingArguments(
    output_dir="polish_bart_base_finetuned/results_bart",
    eval_strategy="steps",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=6,
    fp16=True
)

In [8]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    tokenizer=tokenizer,
    data_collator=data_collator
)

In [9]:
trainer.train()

E:\Python\Projects\PSL-Translator\.venv\lib\site-packages\transformers\models\bart\modeling_bart.py:496: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:455.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


Step,Training Loss,Validation Loss
500,1.837900,0.981107
1000,0.916400,0.732551
1500,0.751600,0.619462
2000,0.641700,0.546202
2500,0.571700,0.505535
3000,0.528000,0.483605
3500,0.483900,0.447455
4000,0.450500,0.422476
4500,0.423700,0.421270
5000,0.408200,0.389120


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_eos_token_id': 2}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_eos_token_id': 2}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generatio

TrainOutput(global_step=11238, training_loss=0.49602327891613035, metrics={'train_runtime': 2725.1364, 'train_samples_per_second': 65.953, 'train_steps_per_second': 4.124, 'total_flos': 2991337291837440.0, 'train_loss': 0.49602327891613035, 'epoch': 6.0})

In [10]:
trainer.save_model("./polish_bart_base_finetuned")

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_eos_token_id': 2}


In [11]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from evaluation import bleu_evaluation
from translation.translation_dataloader import TranslationDataset

data = TranslationDataset(files=("../../../data/translation/dataset_part1.txt", "../../../data/translation/dataset_part2.txt"))
train_iter, val_iter, test_iter = torch.utils.data.random_split(data, [0.9, 0.05, 0.05])

model_name = "./polish_bart_base_finetuned"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

bleu_evaluation(model, tokenizer, test_iter)

E:\Python\Projects\PSL-Translator\.venv\lib\site-packages\nltk\translate\bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
E:\Python\Projects\PSL-Translator\.venv\lib\site-packages\nltk\translate\bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
E:\Python\Projects\PSL-Translator\.venv\lib\site-packages\nltk\translate\bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-g

0.5492413370892631

In [12]:
sentences = ["on pomagać szczupła kobieta", "oni zjeść śniadanie iść na rower", "ja mieć ból brzuch"]
for sentence in sentences:
    inputs = tokenizer.encode(sentence, return_tensors="pt")
    outputs = model.generate(inputs, max_length=50, num_beams=4, early_stopping=True)
    translation = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(sentence)
    print(translation)

on pomagać szczupła kobieta
On pomaga szczupłą kobietę
oni zjeść śniadanie iść na rower
Oni zjedli śniadanie idą na rower
ja mieć ból brzuch
Mam ból brzucha
